In [1]:
import pandas as pd
import pyarrow as pa

from deltalake import DeltaTable
from deltalake import write_deltalake

In [2]:
BRONZE_PATH = "../data/bronze/patient_readings"
SILVER_PATH = "../data/silver/patient_readings"
GOLD_PATH = "../data/gold/ward_summary"

In [3]:
import pandas as pd

pd.read_csv("../data/validated_patient_readings.csv")

,patient_id,ward,heart_rate,oxygen_level,event_time
0,P001,ICU,82,98,2026-07-22 10:00:00
1,P001,ICU,82,98,2026-07-22 10:00:00
2,P002,ICU,110,92,2026-07-22 10:05:00
3,P003,Ward-A,75,99,2026-07-22 10:10:00
4,P004,Ward-A,88,96,2026-07-22 10:15:00
5,P005,ICU,130,89,2026-07-22 10:20:00
6,P006,Ward-B,70,98,2026-07-22 10:25:00
7,P007,Ward-B,95,94,2026-07-22 10:30:00
8,P008,ICU,105,91,2026-07-22 10:35:00
9,P009,Ward-A,65,100,2026-07-22 10:40:00


## Bronze

Stores validated raw patient records exactly as received from the ingestion layer.

In [10]:
from datetime import datetime

bronze_df = pd.read_csv(
    "../data/validated_patient_readings.csv"
)

bronze_df["source"] = "kafka"

bronze_df["ingestion_time"] = str(datetime.now())

write_deltalake(
    BRONZE_PATH,
    bronze_df,
    mode="append"
)

print("Bronze Created")

dt = DeltaTable(BRONZE_PATH)

dt.to_pandas()

Bronze Created


,patient_id,ward,heart_rate,oxygen_level,event_time,source,ingestion_time
0,P001,ICU,82,98,2026-07-22 10:00:00,kafka,2026-07-23 04:46:25.675497
1,P001,ICU,82,98,2026-07-22 10:00:00,kafka,2026-07-23 04:46:25.675497
2,P002,ICU,110,92,2026-07-22 10:05:00,kafka,2026-07-23 04:46:25.675497
3,P003,Ward-A,75,99,2026-07-22 10:10:00,kafka,2026-07-23 04:46:25.675497
4,P004,Ward-A,88,96,2026-07-22 10:15:00,kafka,2026-07-23 04:46:25.675497
5,P005,ICU,130,89,2026-07-22 10:20:00,kafka,2026-07-23 04:46:25.675497
6,P006,Ward-B,70,98,2026-07-22 10:25:00,kafka,2026-07-23 04:46:25.675497
7,P007,Ward-B,95,94,2026-07-22 10:30:00,kafka,2026-07-23 04:46:25.675497
8,P008,ICU,105,91,2026-07-22 10:35:00,kafka,2026-07-23 04:46:25.675497
9,P009,Ward-A,65,100,2026-07-22 10:40:00,kafka,2026-07-23 04:46:25.675497


## silver

Stores cleaned and deduplicated patient records,
keeping the latest record per patient.

In [36]:
bronze_df["event_time"] = pd.to_datetime(
    bronze_df["event_time"]
)


silver_df = (
    bronze_df
    .sort_values("event_time")
    .drop_duplicates(
        subset=["patient_id"],
        keep="last"
    )
    .copy()
)

print("Rows in Silver:", len(silver_df))
silver_df["event_time"] = silver_df["event_time"].astype(str)

Rows in Silver: 9


In [37]:
write_deltalake(
    SILVER_PATH,
    silver_df,
    mode="overwrite"
)

print("Silver Created")

DeltaTable(
    SILVER_PATH
).to_pandas()

Silver Created


,patient_id,ward,heart_rate,oxygen_level,event_time,source,ingestion_time
0,P001,ICU,82,98,2026-07-22 10:00:00,kafka,2026-07-23 04:46:25.675497
1,P002,ICU,110,92,2026-07-22 10:05:00,kafka,2026-07-23 04:46:25.675497
2,P003,Ward-A,75,99,2026-07-22 10:10:00,kafka,2026-07-23 04:46:25.675497
3,P004,Ward-A,88,96,2026-07-22 10:15:00,kafka,2026-07-23 04:46:25.675497
4,P005,ICU,130,89,2026-07-22 10:20:00,kafka,2026-07-23 04:46:25.675497
5,P006,Ward-B,70,98,2026-07-22 10:25:00,kafka,2026-07-23 04:46:25.675497
6,P007,Ward-B,95,94,2026-07-22 10:30:00,kafka,2026-07-23 04:46:25.675497
7,P008,ICU,105,91,2026-07-22 10:35:00,kafka,2026-07-23 04:46:25.675497
8,P009,Ward-A,65,100,2026-07-22 10:40:00,kafka,2026-07-23 04:46:25.675497


## MERGE

Demonstrates Delta Lake upsert functionality using
patient_id as the business key.

In [20]:
updates_df = pd.DataFrame(
    [
        {
            "patient_id": "P002",
            "ward": "ICU",
            "heart_rate": 115,
            "oxygen_level": 90,
            "event_time": "2026-07-22 11:00:00"
        },
        {
            "patient_id": "P011",
            "ward": "Ward-C",
            "heart_rate": 72,
            "oxygen_level": 99,
            "event_time": "2026-07-22 11:05:00"
        }
    ]
)

updates_df

,patient_id,ward,heart_rate,oxygen_level,event_time
0,P002,ICU,115,90,2026-07-22 11:00:00
1,P011,Ward-C,72,99,2026-07-22 11:05:00


In [21]:
from deltalake import DeltaTable
import pyarrow as pa

silver_table = DeltaTable(SILVER_PATH)

silver_table.merge(
    source=pa.Table.from_pandas(updates_df),
    predicate="target.patient_id = source.patient_id",
    source_alias="source",
    target_alias="target"
).when_matched_update_all(
).when_not_matched_insert_all(
).execute()

{'num_source_rows': 2,
 'num_target_rows_inserted': 1,
 'num_target_rows_updated': 1,
 'num_target_rows_deleted': 0,
 'num_target_rows_copied': 8,
 'num_output_rows': 10,
 'num_target_files_scanned': 1,
 'num_target_files_skipped_during_scan': 0,
 'num_target_files_added': 1,
 'num_target_files_removed': 1,
 'execution_time_ms': 102,
 'scan_time_ms': 34,
 'rewrite_time_ms': 0}

In [22]:
merged_df = DeltaTable(
    SILVER_PATH
).to_pandas()

merged_df.sort_values("patient_id")

,patient_id,ward,heart_rate,oxygen_level,event_time,source,ingestion_time
2,P001,ICU,82,98,2026-07-22 10:00:00,kafka,2026-07-23 04:46:25.675497
1,P002,ICU,115,90,2026-07-22 11:00:00,kafka,2026-07-23 04:46:25.675497
3,P003,Ward-A,75,99,2026-07-22 10:10:00,kafka,2026-07-23 04:46:25.675497
4,P004,Ward-A,88,96,2026-07-22 10:15:00,kafka,2026-07-23 04:46:25.675497
5,P005,ICU,130,89,2026-07-22 10:20:00,kafka,2026-07-23 04:46:25.675497
6,P006,Ward-B,70,98,2026-07-22 10:25:00,kafka,2026-07-23 04:46:25.675497
7,P007,Ward-B,95,94,2026-07-22 10:30:00,kafka,2026-07-23 04:46:25.675497
8,P008,ICU,105,91,2026-07-22 10:35:00,kafka,2026-07-23 04:46:25.675497
9,P009,Ward-A,65,100,2026-07-22 10:40:00,kafka,2026-07-23 04:46:25.675497
0,P011,Ward-C,72,99,2026-07-22 11:05:00,NaN,NaN


## gold

Stores aggregated ward-level metrics including average heart rate, average oxygen level, and patient counts.

In [35]:
gold_df = (
    silver_df
    .groupby("ward")
    .agg(
        avg_heart_rate=("heart_rate", "mean"),
        avg_oxygen_level=("oxygen_level", "mean"),
        patient_count=("patient_id", "count"),
        critical_patients=(
            "oxygen_level",
            lambda x: (x < 92).sum()
        )
    )
    .reset_index()
)

write_deltalake(
    GOLD_PATH,
    gold_df,
    mode="overwrite"
)

print("Gold Created")

DeltaTable(
    GOLD_PATH
).to_pandas()

Gold Created


,ward,avg_heart_rate,avg_oxygen_level,patient_count,critical_patients
0,ICU,108.0,92.000000,4,3
1,Ward-A,76.0,98.333333,3,0
2,Ward-B,82.5,96.000000,2,0
3,Ward-C,72.0,99.000000,1,0


In [41]:
print("Bronze Rows:", len(DeltaTable(BRONZE_PATH).to_pandas()))
print("Silver Rows:", len(DeltaTable(SILVER_PATH).to_pandas()))
print("Gold Rows:", len(DeltaTable(GOLD_PATH).to_pandas()))

Bronze Rows: 10
Silver Rows: 9
Gold Rows: 4


## Schema Enforcement

Demonstrates Delta Lake rejecting writes that do not
match the existing table schema.

In [40]:
wrong_schema_df = pd.DataFrame(
    [
        {
            "patient_id": "P999",
            "ward": "ICU",
            "temperature": 39
        }
    ]
)

try:
    write_deltalake(
        SILVER_PATH,
        wrong_schema_df,
        mode="append"
    )

    print("Write succeeded")

except Exception as e:
    print("Schema Enforcement Worked")
    print(type(e).__name__)
    print(e)

wrong_schema_df

Schema Enforcement Worked
SchemaMismatchError
Cannot cast schema, number of fields does not match: 3 vs 7


,patient_id,ward,temperature
0,P999,ICU,39
